In [2]:
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 38.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.8/98.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 14.1 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp312-cp312-linux_x86_64.whl size=659447 sha256=9bf47a2afa31fdb47240e242c72031992f0ccd6d2a03067085c87edf35060864
  Stored in directory: /root/.cache/pip/wheels/90/2a/71/75ec0cc316cc0ff494bfffa2935e02580129cb7f859a0cfd8f
Successfully built pycuda


In [3]:
import numpy as np
import pycuda.autoinit  # инициализация контекста
import pycuda.driver as drv
from pycuda.compiler import SourceModule
import time


def check_shapes(a: np.ndarray, b: np.ndarray):
    if a.shape[1] != b.shape[0]:
        raise ValueError("Incompatible shapes for multiplication")


def multiply_cpu(a: np.ndarray, b: np.ndarray):
    check_shapes(a, b)

    rows, inner = a.shape
    _, cols = b.shape

    res = np.zeros((rows, cols), dtype=np.float32)

    t0 = time.time()

    for i in range(rows):
        for j in range(cols):
            acc = 0.0
            for k in range(inner):
                acc += a[i, k] * b[k, j]
            res[i, j] = acc

    t1 = time.time()
    return res, t1 - t0


def build_kernel():
    return SourceModule("""
    __global__ void matmul(float* A, float* B, float* C, int rows, int cols, int inner) {
        int x = blockIdx.x * blockDim.x + threadIdx.x;
        int y = blockIdx.y * blockDim.y + threadIdx.y;

        if (y < rows && x < cols) {
            float val = 0.0;
            for (int i = 0; i < inner; ++i) {
                val += A[y * inner + i] * B[i * cols + x];
            }
            C[y * cols + x] = val;
        }
    }
    """)


def multiply_gpu(a: np.ndarray, b: np.ndarray):
    check_shapes(a, b)

    rows, inner = a.shape
    _, cols = b.shape

    kernel_module = build_kernel()
    kernel_func = kernel_module.get_function("matmul")

    # выделение памяти
    a_dev = drv.mem_alloc(a.nbytes)
    b_dev = drv.mem_alloc(b.nbytes)
    c_dev = drv.mem_alloc(rows * cols * a.dtype.itemsize)

    drv.memcpy_htod(a_dev, a)
    drv.memcpy_htod(b_dev, b)

    block = (16, 16, 1)
    grid = (
        int(np.ceil(cols / block[0])),
        int(np.ceil(rows / block[1])),
        1
    )

    t0 = time.time()

    kernel_func(
        a_dev, b_dev, c_dev,
        np.int32(rows), np.int32(cols), np.int32(inner),
        block=block,
        grid=grid
    )

    drv.Context.synchronize()
    t1 = time.time()

    result = np.empty((rows, cols), dtype=np.float32)
    drv.memcpy_dtoh(result, c_dev)

    return result, t1 - t0


def run_test():
    size = (50, 50)

    A = np.random.rand(*size).astype(np.float32)
    B = np.random.rand(*size).astype(np.float32)

    cpu_res, cpu_time = multiply_cpu(A, B)
    gpu_res, gpu_time = multiply_gpu(A, B)

    print(f"CPU time: {cpu_time:.6f} sec")
    print(f"GPU time: {gpu_time:.6f} sec")


if __name__ == "__main__":
    run_test()

CPU time: 0.047088 sec
GPU time: 0.001089 sec


In [ ]:
def benchmark():
    dimensions = [50, 100, 200, 400, 800, 1600, 3200, 6400, 12800]

    for dim in dimensions:
        rows = cols = dim

        A = np.random.rand(rows, cols).astype(np.float32)
        B = np.random.rand(rows, cols).astype(np.float32)

        print(f"\n=== Matrix size: {rows} x {cols} ===")

        # CPU
        _, cpu_time = multiply_cpu(A, B)
        print(f"[CPU] elapsed: {cpu_time:.4f} s")

        # GPU
        _, gpu_time = multiply_gpu(A, B)
        print(f"[GPU] elapsed: {gpu_time:.4f} s")

        if gpu_time > 0:
            ratio = cpu_time / gpu_time
        else:
            ratio = float("inf")

        print(f"[Speedup] x{ratio:.2f}")


if __name__ == "__main__":
    benchmark()


=== Matrix size: 50 x 50 ===
[CPU] elapsed: 0.0499 s
[GPU] elapsed: 0.0001 s
[Speedup] x573.45

=== Matrix size: 100 x 100 ===
[CPU] elapsed: 0.3792 s
[GPU] elapsed: 0.0001 s
[Speedup] x4762.19

=== Matrix size: 200 x 200 ===
[CPU] elapsed: 2.8524 s
[GPU] elapsed: 0.0001 s
[Speedup] x20952.24

=== Matrix size: 400 x 400 ===
[CPU] elapsed: 25.1882 s
[GPU] elapsed: 0.0005 s
[Speedup] x46850.17

=== Matrix size: 800 x 800 ===
[CPU] elapsed: 203.3268 s
[GPU] elapsed: 0.0045 s
[Speedup] x45651.43

=== Matrix size: 1600 x 1600 ===
[CPU] elapsed: 1638.6812 s
[GPU] elapsed: 0.0356 s
[Speedup] x46073.95

=== Matrix size: 3200 x 3200 ===
